# Matrice de faisabilité patient × hôpital

Ce notebook construit les **contraintes de ressources** : pour chaque patient et chaque hôpital, on vérifie si les capacités (lits, personnel, O₂, etc.) couvrent les besoins exprimés dans le fichier patients.

**Entrées**
- Fichier patients (besoins par colonne, ex. `Medecins`, `O2`, `Lits_totaux`).
- Fichier hôpitaux `Book1.xlsx` (capacités avec les noms de colonnes côté `'h'` du mapping).
- `resource_mapping.py` : correspondance `besoin patient → colonne hôpital`.

**Sorties** (dossier `output/`)
- `feasibility_matrix.csv` : booléens, une ligne par patient, une colonne par hôpital.
- `feasibility_long.csv` : format long (`patient_id`, `hospital`, `feasible`).
- `feasibility_summary.csv` : nombre de sites faisables par patient.

> **Note** : exécuter les cellules avec le répertoire de travail courant sur le dossier `deep_neural_network` (là où se trouvent ce notebook et `resource_mapping.py`). Sinon, adaptez la variable `ROOT` dans la cellule de configuration.

## 1. Imports et chemins des fichiers

- `pandas` / `openpyxl` : lecture des Excel.
- `defaultdict` : accumulation des besoins quand plusieurs types de besoin se traduisent par la **même** colonne hôpital (ex. `Moniteurs` + `Monitoring` → capacité `Moniteurs`).

In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from resource_mapping import RESOURCE_MAPPINGS

# Répertoire du projet (notebook + resource_mapping.py). Si besoin, remplacez par un chemin absolu.
ROOT = Path.cwd().resolve()
PATIENTS_XLSX = ROOT.parent / "xgboost_model" / "patients_1000_ULTRA_COMPLET.xlsx"
HOSPITALS_XLSX = ROOT / "Book1.xlsx"
OUT_DIR = ROOT / "output"

## 2. Utilitaires et règles métier

1. **`_num`** : convertit une valeur Excel en nombre (valeurs manquantes ou invalides → `0`).
2. **`patient_needs_by_hospital_column`** : pour chaque colonne de capacité hôpital, **somme** tous les besoins patients qui s’y rattachent via `RESOURCE_MAPPINGS`.
3. **`hospital_can_satisfy`** : pour chaque besoin strictement positif, la capacité hôpital sur cette colonne doit être **≥** au besoin ; sinon l’affectation est **non faisable**.
4. **`build_feasibility`** : double boucle patients × hôpitaux, puis agrégation pour le résumé.

In [2]:
def _num(x) -> float:
    if pd.isna(x):
        return 0.0
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0


def patient_needs_by_hospital_column(patient: pd.Series) -> dict[str, float]:
    """Somme les besoins patient qui pointent vers la même colonne hôpital (ex. Moniteurs + Monitoring)."""
    acc: dict[str, float] = defaultdict(float)
    for spec in RESOURCE_MAPPINGS.values():
        p_col, h_col = spec["p"], spec["h"]
        if p_col not in patient.index:
            continue
        acc[h_col] += _num(patient[p_col])
    return dict(acc)


def hospital_can_satisfy(hospital: pd.Series, needs: dict[str, float]) -> bool:
    for h_col, need in needs.items():
        if need <= 0:
            continue
        cap = _num(hospital[h_col]) if h_col in hospital.index else 0.0
        if cap < need:
            return False
    return True


def build_feasibility(
    patients: pd.DataFrame,
    hospitals: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    hosp_names = hospitals["Hôpital"].astype(str).tolist()
    rows: list = []
    long_rows: list[dict] = []

    for _, p in patients.iterrows():
        pid = p["ID"]
        needs = patient_needs_by_hospital_column(p)
        flags = []
        for _, h in hospitals.iterrows():
            ok = hospital_can_satisfy(h, needs)
            flags.append(ok)
            long_rows.append({"patient_id": pid, "hospital": h["Hôpital"], "feasible": int(ok)})
        rows.append([pid, *flags])

    matrix = pd.DataFrame(
        [r[1:] for r in rows],
        index=[r[0] for r in rows],
        columns=hosp_names,
    )
    matrix.index.name = "patient_id"

    long_df = pd.DataFrame(long_rows)
    summary = (
        long_df.groupby("patient_id")["feasible"]
        .agg(n_feasible="sum", n_hospitals="count")
        .reset_index()
    )
    summary["frac_feasible"] = summary["n_feasible"] / summary["n_hospitals"].replace(0, np.nan)

    return matrix, summary

## 3. Chargement des tables Excel

Vérification minimale : présence de la colonne **`Hôpital`** dans la table des établissements.

In [3]:
patients = pd.read_excel(PATIENTS_XLSX)
hospitals = pd.read_excel(HOSPITALS_XLSX)

if "Hôpital" not in hospitals.columns:
    raise KeyError("La feuille hôpitaux doit contenir la colonne « Hôpital ».")

print(f"Patients : {len(patients)} lignes, {patients.shape[1]} colonnes")
print(f"Hôpitaux : {len(hospitals)} lignes, {hospitals.shape[1]} colonnes")

Patients : 1000 lignes, 36 colonnes
Hôpitaux : 22 lignes, 26 colonnes


## 4. Calcul, export CSV et aperçu

Les fichiers sont écrits dans `output/`. Le résumé affiche combien de patients ont **au moins un** hôpital réaliste au sens des ressources.

In [4]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

matrix, summary = build_feasibility(patients, hospitals)

matrix.to_csv(OUT_DIR / "feasibility_matrix.csv")
summary.to_csv(OUT_DIR / "feasibility_summary.csv", index=False)

long = matrix.reset_index().melt(
    id_vars=["patient_id"],
    var_name="hospital",
    value_name="feasible",
)
long["feasible"] = long["feasible"].astype(int)
long.to_csv(OUT_DIR / "feasibility_long.csv", index=False)

n_pat, n_h = matrix.shape
feasible_any = int((matrix.sum(axis=1) > 0).sum())
print(f"Patients: {n_pat}, hôpitaux: {n_h}")
print(
    f"Patients avec au moins un hôpital faisable: {feasible_any} ({100 * feasible_any / n_pat:.1f}%)"
)
print(f"Fichiers écrits dans: {OUT_DIR.resolve()}")

display(summary.head(10))
display(matrix.iloc[:5, :6])

Patients: 1000, hôpitaux: 22
Patients avec au moins un hôpital faisable: 1000 (100.0%)
Fichiers écrits dans: C:\Users\HP\OneDrive\Desktop\silma\ml_models_for_patients_allocations\deep_neural_network\output


,patient_id,n_feasible,n_hospitals,frac_feasible
0,1,17,22,0.772727
1,2,20,22,0.909091
2,3,17,22,0.772727
3,4,17,22,0.772727
4,5,6,22,0.272727
5,6,20,22,0.909091
6,7,6,22,0.272727
7,8,6,22,0.272727
8,9,6,22,0.272727
9,10,20,22,0.909091


,Centre Hospitalier Universitaire Yalgado Ouédraogo,Centre Hospitalier Universitaire de Tingandogo,Centre Hospitalier Universitaire de Bogodogo,CHU Pédiatrique Charles de Gaulle,Hôpital Paul VI,Hôpital Saint Camille
patient_id,,,,,,
1,True,True,True,True,True,True
2,True,True,True,True,True,True
3,True,True,True,True,True,True
4,True,True,True,True,True,True
5,True,True,True,True,False,False
